In [1]:
import cv2
import numpy as np
import time

from ultralytics import YOLO
from ultralytics.utils.plotting import colors

from collections import defaultdict

In [2]:
class ObjectTracking:
    """Object Tracking using Ultralytics YOLO26: https://docs.ultralytics.com/models/yolo26/"""

    def __init__(self, model_path="yolo26n.pt", source="path/to/video.mp4"):

        self.model = YOLO(model_path)  # Model initialization
        self.names = self.model.names  # Class names

        # Video capturing module
        self.cap = cv2.VideoCapture(source)
        assert self.cap.isOpened(), "Error reading video file"

        # Video writing module
        w, h, fps = (
            int(self.cap.get(x)) 
            for x in (
                cv2.CAP_PROP_FRAME_WIDTH, 
                cv2.CAP_PROP_FRAME_HEIGHT, 
                cv2.CAP_PROP_FPS))
        self.writer = cv2.VideoWriter(
            "object-tracking.avi",
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps, 
            (w, h)
            )

        self.track_history = defaultdict(lambda: [])  # Store the track history

        # Display settings
        self.rect_width=2
        self.font = 0.5
        self.text_width=2
        self.padding = 12
        self.margin = 10
        self.circle_thickness=5
        self.polyline_thickness=2

        # Window setup
        self.window_name = "YOLO Tracking"
        cv2.namedWindow(self.window_name, cv2.WINDOW_NORMAL)

    def draw_bbox(self, im0, box, track_id, cls):
        """Draw bounding box with label at TOP-LEFT, but TEXT CENTERED in its box."""

        x1, y1, x2, y2 = map(int, box)

        color = colors(int(cls), True)

        # Draw main bounding box
        cv2.rectangle(im0, (x1, y1), (x2, y2), color, self.rect_width)

        # Prepare label
        label = f"{self.names[int(cls)]}:{int(track_id)}"

        # Get text size
        (tw, th), _ = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, self.font, self.text_width
        )

        bg_x1 = x1  # left edge of bbox
        bg_x2 = bg_x1 + (tw + 2 * self.padding)

        bg_y2 = y1  # top of bbox
        bg_y1 = bg_y2 - (th + 2 * self.margin)

        # Draw filled background rectangle (top-left)
        cv2.rectangle(
            im0,
            (bg_x1, bg_y1),
            (bg_x2, bg_y2),
            color,
            -1,
        )

        text_x = bg_x1 + ((bg_x2 - bg_x1) - tw) // 2
        text_y = bg_y1 + ((bg_y2 - bg_y1) + th) // 2 - 2  # small vertical tweak

        cv2.putText(
            im0,
            label,
            (text_x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            self.font,
            (104, 31, 17) if cls==2 else (255, 255, 255) ,  # white text
            self.text_width,
            cv2.LINE_AA,
        )

    def run(self, conf=0.1, iou=0.45, tracker="bytetrack.yaml", imgsz=960):
        """Lancement du tracking avec paramètres variables."""
        print(f"Test en cours : Conf={conf}, IoU={iou}, Tracker={tracker}")
        
        while self.cap.isOpened():
            success, im0 = self.cap.read()
            if not success: break

            results = self.model.track(
                im0, 
                persist=True,
                tracker=tracker,
                conf=conf,
                iou=iou,
                imgsz=imgsz,
                verbose=False, # Désactive le flux de texte pour plus de clarté
                classes=[0, 1, 3, 4]
            )

            if results and len(results) > 0:
                result = results[0]

                if result.boxes is not None and result.boxes.id is not None:
                    boxes = result.boxes.xyxy.cpu()
                    ids = result.boxes.id.cpu()
                    clss = result.boxes.cls.tolist()

                    if boxes is not None or ids is not None:
                        for box, id, cls in zip(boxes, ids.tolist(), clss):
                            self.draw_bbox(im0, box, id, cls)

                            x1, y1, x2, y2 = box
                            track = self.track_history[id]

                            # append box centroid
                            track.append(
                                (float((x1+x2)/2), 
                                float((y1+y2)/2))
                                )  
                            if len(track) > 50:  # retain 30 tracks for 30 frames
                                track.pop(0)
    
                            # draw the tracking lines
                            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                            
                            cv2.circle(
                                im0,
                                (int(track[-1][0]), int(track[-1][1])),
                                5,
                                colors(cls, True),
                                -1
                            )

                            cv2.polylines(
                                im0, 
                                [points], 
                                isClosed=False, 
                                color=colors(cls, True), 
                                thickness=self.polyline_thickness
                                )

            self.writer.write(im0)
            cv2.imshow(self.window_name, im0)  # Display and handle input

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('c'):
                self.selected_id = None
                print("Selection cleared")

        # Cleanup
        self.cap.release()
        cv2.destroyAllWindows()

In [3]:
def train_model(epochs, imgsz, model_name):
    model = YOLO(model_name) 
    model.train(
        data='data/data.yaml', 
        epochs=epochs, 
        imgsz=imgsz, 
        batch=16,
        device=0,     
        amp=True,
        patience=20,
        #mosaic=1.0,
        scale=0.5,
        hsv_v=0.4,
        workers=4
    )

In [4]:
epochs = 150
imgsz = 960    # Multiple de 32 le plus proche de 1080
model_name = "yolo26m.pt"
train_model(epochs, imgsz, model_name)

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train8, nbs=

In [5]:
test_configs = [
    {"conf": 0.05, "iou": 1, "tracker": "bytetrack.yaml", "imgsz": 960},
    #{"conf": 0.10, "iou": 0.50, "tracker": "bytetrack.yaml"},
    #{"conf": 0.15, "iou": 0.45, "tracker": "botsort.yaml"},
]

for i, config in enumerate(test_configs):
    start_time = time.time()
    # Initialise un nouveau tracker pour chaque test
    tracker = ObjectTracking(
        model_path="runs/detect/train8/weights/best.pt", 
        source="videos/video_test.mp4"
    )
    # Modifie le nom du fichier de sortie pour ne pas l'écraser
    tracker.writer = cv2.VideoWriter(
        f"test_result_{"medium"}.avi",
        cv2.VideoWriter_fourcc(*"mp4v"),
        tracker.cap.get(cv2.CAP_PROP_FPS),
        (int(tracker.cap.get(3)), int(tracker.cap.get(4)))
    )
    
    tracker.run(**config)
    end_time = time.time()
    print(f"Test {i} completed in {end_time - start_time:.2f} seconds")


Test en cours : Conf=0.05, IoU=1, Tracker=bytetrack.yaml
Test 0 completed in 23.50 seconds


In [ ]:
# "runs/detect/train/weights/best.pt" est le chemin par défaut après train
tracker = ObjectTracking(model_path="runs/detect/train5/weights/best.pt", source="videos/video_basket.mp4")
tracker.run()